# Heritage Short: Wan 2.2 TI2V-5B image-to-video (Kaggle)

Turns keyframes into ~5 s clips. Checked against the diffusers 0.43.0 source
(`WanImageToVideoPipeline` with `expand_timesteps=True` is the Wan 2.2 TI2V mode), but
**not yet run on a GPU**: watch the first clip and read the log before running more.

**Before you start**
1. On your machine: `make kaggle-export` → `kaggle/heritage_job.zip`.
2. Kaggle → *Create → New Dataset* → upload the zip → name it `heritage-job` → *Create*.
3. Kaggle → *Create → New Notebook* → *File → Import Notebook* → this file.
4. Right panel: *Add Input* → your `heritage-job` dataset. *Session options*: Accelerator **GPU T4 x2**,
   Internet **On** (needs a phone-verified account), Persistence **Files only** (keeps finished clips if the session restarts).
5. *Run All*. The first run downloads about 20 GB of weights; then each clip takes several minutes on a T4.
6. When it finishes: right panel → *Output* → `/kaggle/working/heritage_results_wan.zip` → download.
   On your machine: `make import ZIP=~/Downloads/heritage_results_wan.zip`.

Clips are saved one by one and finished ones are skipped, so you can stop and re-run at any time.

In [ ]:
# 1. GPU check

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout or "NO GPU: Settings -> Accelerator -> GPU T4 x2")

In [ ]:
# 2. Install: pinned diffusers release that contains the Wan 2.2 TI2V pipeline
!pip install -q "diffusers==0.43.0" "transformers>=4.51" accelerate ftfy imageio imageio-ffmpeg

In [ ]:
# 3. Job, output folder and logger (same JSON format as the local pipeline)

import glob, json, os, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

found = glob.glob("/kaggle/input/**/jobs.json", recursive=True)
assert found, "jobs.json not found: attach the heritage-job dataset (Add input) and re-run this cell"
JOB = Path(found[0]).parent
jobs = json.loads((JOB / "jobs.json").read_text())
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "runs_kaggle.jsonl"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-kaggle"

def gpu_name():
    try:
        import torch
        return torch.cuda.get_device_name(0)
    except Exception:
        return "unknown"

def log_call(stage, shot, model, params, seed, latency_s, status, output=None, error=None):
    """Same shape as pipeline/common.py log lines, so `make import` can merge them."""
    rec = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds"), "run_id": RUN_ID,
           "event": "model_call", "stage": stage, "shot": shot, "provider": "kaggle", "model": model,
           "params": params, "seed": seed, "output": output, "attempt": 1, "status": status,
           "latency_s": round(latency_s, 2), "gpu": gpu_name(),
           "cost": {"gpu_minutes": round(latency_s / 60, 2), "tier": "free (Kaggle weekly GPU quota)"}}
    if error:
        rec["error"] = str(error)[:300]
    with LOG.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"{status:5} {shot or ''} {latency_s:7.1f}s {output or error or ''}")

print("job folder:", JOB, "| storyboard version", jobs["storyboard_version"])

In [ ]:
# 4. Which shots to animate, and how many per session
# Default: the non-character shots whose motion a still cannot show (steam, mist).
# Character shots (s04, s06, s07) are left out on purpose: image-to-video may turn her face
# to the camera, and likeness consent is still pending. Add them only after consent.
SHOTS = ["s01", "s02", "s08", "s09"]
MAX_CLIPS = 4            # stop after this many new clips in one session
STEPS = 30               # jobs.json default; the model card uses 50 (slower, a bit better)
todo = [j for j in jobs["wan"] if j["shot"] in SHOTS]
todo.sort(key=lambda j: SHOTS.index(j["shot"]))
for j in todo:
    print(j["shot"], "|", j["prompt"])

In [ ]:
# 5. Load the model. A T4 has no bfloat16, so the transformer runs in float16 (VAE in float32)
# with model CPU offload so it fits in 16 GB.
import numpy as np, torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

P = jobs["wan_params"]
DTYPE = torch.float16
vae = AutoencoderKLWan.from_pretrained(P["model"], subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(P["model"], vae=vae, torch_dtype=DTYPE, expand_timesteps=True)
pipe.enable_model_cpu_offload()
mod = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]
print("loaded", P["model"], "| expand_timesteps =", pipe.config.expand_timesteps, "| size multiple", mod, "|", gpu_name())

In [ ]:
# 6. Generate. Each clip is checked for black or NaN frames (a known float16 risk on T4)
(OUT / "clips").mkdir(exist_ok=True)
done = 0
for job in todo:
    out = OUT / "clips" / job["out"]
    if out.exists():
        print("skip (done)", job["shot"]); continue
    if done >= MAX_CLIPS:
        break
    image = load_image(str(JOB / job["keyframe"]))
    ratio = image.height / image.width
    h = round(np.sqrt(P["max_area"] * ratio)) // mod * mod
    w = round(np.sqrt(P["max_area"] / ratio)) // mod * mod
    image = image.resize((w, h))
    params = {"width": w, "height": h, "num_frames": P["num_frames"], "steps": STEPS,
              "guidance_scale": P["guidance_scale"], "dtype": str(DTYPE), "prompt": job["prompt"]}
    t0 = time.perf_counter()
    try:
        frames = pipe(image=image, prompt=job["prompt"], negative_prompt=P["negative_prompt"],
                      height=h, width=w, num_frames=P["num_frames"], guidance_scale=P["guidance_scale"],
                      num_inference_steps=STEPS,
                      generator=torch.Generator("cpu").manual_seed(job["seed"])).frames[0]
        arr = np.asarray(frames, dtype=np.float32)
        if np.isnan(arr).any() or arr.max() < 0.02:
            raise RuntimeError("black or NaN frames (float16 overflow); this shot stays Ken Burns")
        export_to_video(frames, str(out), fps=P["fps"])
        log_call("shots", job["shot"], P["model"], params, job["seed"], time.perf_counter() - t0, "ok",
                 output=f"outputs/shots/wan/{job['out']}")
        done += 1
    except Exception as err:   # OOM, NaN: log it and move on; Ken Burns covers this shot
        log_call("shots", job["shot"], P["model"], params, job["seed"], time.perf_counter() - t0, "error", error=err)
        torch.cuda.empty_cache()

In [ ]:
# 7. Watch the first clip here before trusting the rest
from IPython.display import Video, display
for f in sorted((OUT / "clips").glob("*.mp4"))[:1]:
    display(Video(str(f), embed=True, width=360))

In [ ]:
# 8. Zip results for download
ZIP_NAME = "heritage_results_wan.zip"

# One download brings everything home. On your machine: make import ZIP=~/Downloads/<this file>
zpath = Path("/kaggle/working") / ZIP_NAME
with zipfile.ZipFile(zpath, "w") as z:
    for f in OUT.rglob("*"):
        if f.is_file():
            z.write(f, f.relative_to(OUT.parent))
print("download:", zpath, round(zpath.stat().st_size / 1e6, 1), "MB")
print(open(LOG).read() if LOG.exists() else "no log lines")